In [ ]:
!pip install pgeocode
!pip install plotly
!pip install census us
!pip install geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 6.9 MB/s eta 0:00:00


In [ ]:
import plotly.express as px
from google.colab import files
import pandas as pd
import pgeocode
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from census import Census
import geopandas as gpd

In [ ]:


uploaded = files.upload()

Saving services2024.xlsx to services2024.xlsx


In [ ]:
import pandas as pd

service = pd.read_excel("services2024.xlsx")

service_ca = service[service['state'] == 'CA']
service_ca.head()

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,name1,name2,street1,street2,city,state,zip,phone,intake1,intake2,intake1a,intake2a,service_code_info
626,BNI Treatment Centers,NaN,30954 Lobo Canyon Road,NaN,Agoura Hills,CA,91301,866-801-0085,NaN,NaN,NaN,NaN,SA MH SUMH * RES * RTCC * ARIPI CLOZA LURAS OL...
627,Felton Institute,OAST reMIND BEAM,1005 Atlantic Avenue,NaN,Alameda,CA,94501,510-844-8244,NaN,NaN,NaN,NaN,MH SUMH * OP * CMHC * HALOP PERPH ARIPI CLOZA ...
628,College Hospital Cerritos,Anaheim PHP,1488 East Lincoln Avenue,NaN,Anaheim,CA,92805,714-776-2500,714-493-6354,NaN,NaN,NaN,SA MH SUMH * OP PHDT * PH * FLUPH HALOP ARIPI ...
629,Newport Academy,NaN,195 South Peralta Hills Drive,NaN,Anaheim,CA,92807,877-820-6371,NaN,NaN,NaN,NaN,SA MH SUMH * OP PHDT RES * RTCC * AT CBT CFT D...
630,Crestwood Behavioral Health Inc,Crestwood Center at Napa Valley,295 Pine Breeze Drive,NaN,Angwin,CA,94508,707-965-2461,NaN,NaN,NaN,NaN,MH * HI RES * RTCA * FLUPH HALOP NRT NSC ANTPY...


In [ ]:
nomi = pgeocode.Nominatim('us')

service_ca['county'] = service_ca['zip'].astype(str).str.zfill(5).apply(
    lambda z: nomi.query_postal_code(z)['county_name']
)

service_ca['county'].value_counts()
service_ca.head()

/tmp/ipykernel_33814/388114722.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  service_ca['county'] = service_ca['zip'].astype(str).str.zfill(5).apply(


,name1,name2,street1,street2,city,state,zip,phone,intake1,intake2,intake1a,intake2a,service_code_info,county
626,BNI Treatment Centers,NaN,30954 Lobo Canyon Road,NaN,Agoura Hills,CA,91301,866-801-0085,NaN,NaN,NaN,NaN,SA MH SUMH * RES * RTCC * ARIPI CLOZA LURAS OL...,Los Angeles
627,Felton Institute,OAST reMIND BEAM,1005 Atlantic Avenue,NaN,Alameda,CA,94501,510-844-8244,NaN,NaN,NaN,NaN,MH SUMH * OP * CMHC * HALOP PERPH ARIPI CLOZA ...,Alameda
628,College Hospital Cerritos,Anaheim PHP,1488 East Lincoln Avenue,NaN,Anaheim,CA,92805,714-776-2500,714-493-6354,NaN,NaN,NaN,SA MH SUMH * OP PHDT * PH * FLUPH HALOP ARIPI ...,Orange
629,Newport Academy,NaN,195 South Peralta Hills Drive,NaN,Anaheim,CA,92807,877-820-6371,NaN,NaN,NaN,NaN,SA MH SUMH * OP PHDT RES * RTCC * AT CBT CFT D...,Orange
630,Crestwood Behavioral Health Inc,Crestwood Center at Napa Valley,295 Pine Breeze Drive,NaN,Angwin,CA,94508,707-965-2461,NaN,NaN,NaN,NaN,MH * HI RES * RTCA * FLUPH HALOP NRT NSC ANTPY...,Napa


In [ ]:
c = Census("972a9abf4608d7bcdea7493682c40e8d3a5d2893")

pop_data = c.acs5.state_county(
    fields=('NAME', 'B01003_001E'),
    state_fips='06',
    county_fips='*'
)

pop_df = pd.DataFrame(pop_data)
pop_df.columns = ['name', 'population', 'state', 'county_fips']

pop_df['fips'] = pop_df['state'] + pop_df['county_fips']
pop_df['population'] = pop_df['population'].astype(int)

pop_df.head()

,name,population,state,county_fips,fips
0,"Alameda County, California",1649473,06,001,06001
1,"Alpine County, California",1616,06,003,06003
2,"Amador County, California",41428,06,005,06005
3,"Butte County, California",207929,06,007,06007
4,"Calaveras County, California",46248,06,009,06009


In [ ]:
service_ca = service_ca.copy()

nomi = pgeocode.Nominatim('us')

service_ca['fips'] = '06' + service_ca['zip'].astype(str).str.zfill(5).apply(
    lambda z: str(int(nomi.query_postal_code(z)['county_code'])).zfill(3)
)

service_ca['county'] = service_ca['zip'].astype(str).str.zfill(5).apply(
    lambda z: nomi.query_postal_code(z)['county_name']
)

fips_counts = service_ca['fips'].value_counts().reset_index()
fips_counts.columns = ['fips', 'count']

fips_to_county = service_ca[['fips', 'county']].drop_duplicates().set_index('fips')['county']
fips_counts['county'] = fips_counts['fips'].map(fips_to_county)

fips_counts = fips_counts.merge(pop_df, on='fips', how='left')

fips_counts['rate'] = (fips_counts['count'] / fips_counts['population']) * 100000

service_ca = service_ca.merge(pop_df, on='fips', how='left')
service_ca.head()

,name1,name2,street1,street2,city,state_x,zip,phone,intake1,intake2,intake1a,intake2a,service_code_info,county,fips,name,population,state_y,county_fips
0,BNI Treatment Centers,NaN,30954 Lobo Canyon Road,NaN,Agoura Hills,CA,91301,866-801-0085,NaN,NaN,NaN,NaN,SA MH SUMH * RES * RTCC * ARIPI CLOZA LURAS OL...,Los Angeles,06037,"Los Angeles County, California",9808667,06,037
1,Felton Institute,OAST reMIND BEAM,1005 Atlantic Avenue,NaN,Alameda,CA,94501,510-844-8244,NaN,NaN,NaN,NaN,MH SUMH * OP * CMHC * HALOP PERPH ARIPI CLOZA ...,Alameda,06001,"Alameda County, California",1649473,06,001
2,College Hospital Cerritos,Anaheim PHP,1488 East Lincoln Avenue,NaN,Anaheim,CA,92805,714-776-2500,714-493-6354,NaN,NaN,NaN,SA MH SUMH * OP PHDT * PH * FLUPH HALOP ARIPI ...,Orange,06059,"Orange County, California",3165820,06,059
3,Newport Academy,NaN,195 South Peralta Hills Drive,NaN,Anaheim,CA,92807,877-820-6371,NaN,NaN,NaN,NaN,SA MH SUMH * OP PHDT RES * RTCC * AT CBT CFT D...,Orange,06059,"Orange County, California",3165820,06,059
4,Crestwood Behavioral Health Inc,Crestwood Center at Napa Valley,295 Pine Breeze Drive,NaN,Angwin,CA,94508,707-965-2461,NaN,NaN,NaN,NaN,MH * HI RES * RTCA * FLUPH HALOP NRT NSC ANTPY...,Napa,06055,"Napa County, California",134869,06,055


In [ ]:
crimes = pd.read_csv('crimes.csv', low_memory=False)
crimes['County'] = crimes['County'].str.replace(' County', '', regex=False)

county_fips_map = service_ca[['county', 'fips']].drop_duplicates().set_index('county')['fips']

crimes['fips'] = crimes['County'].map(county_fips_map)

crimes = crimes.merge(pop_df, on='fips', how='left')

crimes_10yr = crimes[crimes['Year'].between(2015, 2024)]
crimes_avg = crimes_10yr.groupby(['County', 'fips', 'population'])['Violent_sum'].mean().reset_index()
crimes_avg.columns = ['County', 'fips', 'population', 'avg_violent_sum']

crimes_avg['avg_crime_rate'] = (crimes_avg['avg_violent_sum'] / crimes_avg['population']) * 100000

crimes_avg.head()

,County,fips,population,avg_violent_sum,avg_crime_rate
0,Alameda,06001,1649473.0,528.603687,32.046823
1,Alpine,06003,1616.0,7.235294,447.728596
2,Amador,06005,41428.0,19.915254,48.071966
3,Butte,06007,207929.0,110.867347,53.319810
4,Calaveras,06009,46248.0,35.941176,77.714012


In [ ]:
merged = service_ca.merge(crimes, left_on='county', right_on='County', how='left')
merged.head()

,name1,name2,street1,street2,city,state_x,zip,phone,intake1,intake2,...,LT400nao_sum,LT200400nao_sum,LT200nao_sum,LT50200nao_sum,LT50nao_sum,fips_y,name_y,population_y,state,county_fips_y
0,BNI Treatment Centers,NaN,30954 Lobo Canyon Road,NaN,Agoura Hills,CA,91301,866-801-0085,NaN,NaN,...,598.0,359.0,NaN,515.0,607.0,06037,"Los Angeles County, California",9808667.0,06,037
1,BNI Treatment Centers,NaN,30954 Lobo Canyon Road,NaN,Agoura Hills,CA,91301,866-801-0085,NaN,NaN,...,347.0,269.0,NaN,406.0,325.0,06037,"Los Angeles County, California",9808667.0,06,037
2,BNI Treatment Centers,NaN,30954 Lobo Canyon Road,NaN,Agoura Hills,CA,91301,866-801-0085,NaN,NaN,...,51.0,49.0,NaN,83.0,54.0,06037,"Los Angeles County, California",9808667.0,06,037
3,BNI Treatment Centers,NaN,30954 Lobo Canyon Road,NaN,Agoura Hills,CA,91301,866-801-0085,NaN,NaN,...,22.0,13.0,NaN,32.0,21.0,06037,"Los Angeles County, California",9808667.0,06,037
4,BNI Treatment Centers,NaN,30954 Lobo Canyon Road,NaN,Agoura Hills,CA,91301,866-801-0085,NaN,NaN,...,205.0,189.0,NaN,374.0,469.0,06037,"Los Angeles County, California",9808667.0,06,037


In [ ]:
url = "https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/tl_2023_us_county.zip"
ca_counties = gpd.read_file(url)

ca_counties = ca_counties[ca_counties['STATEFP'] == '06']

ca_counties['sq_miles'] = ca_counties['ALAND'] / 2589988.11

ca_counties[['NAME', 'GEOID', 'sq_miles']].head()

,NAME,GEOID,sq_miles
8,Sierra,06091,953.168305
324,Sacramento,06067,965.279723
328,Santa Barbara,06083,2733.941097
345,Calaveras,06009,1020.016122
393,Ventura,06111,1840.774860


In [ ]:
merged = merged.rename(columns={'fips_x': 'fips', 'population_x': 'population'})

county_summary = merged.groupby('fips').agg(
    county=('county', 'first'),
    facility_count=('name1', 'nunique'),
    population=('population', 'first'),
).reset_index()

POP_THRESHOLD = 75000
county_summary = county_summary[county_summary['population'] >= POP_THRESHOLD]
crimes_avg_clean = crimes_avg[crimes_avg['population'] >= POP_THRESHOLD]

county_summary = county_summary.merge(
    crimes_avg_clean[['fips', 'avg_crime_rate']], on='fips', how='left'
)

ca_counties_area = ca_counties.rename(columns={'GEOID': 'fips'})[['fips', 'sq_miles']]
county_summary = county_summary.merge(ca_counties_area, on='fips', how='left')

county_summary['service_rate'] = (county_summary['facility_count'] / county_summary['population']) * 100000
county_summary['facility_density'] = county_summary['facility_count'] / county_summary['sq_miles']

geojson_url = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"

fig = px.choropleth(
    county_summary,
    geojson=geojson_url,
    locations='fips',
    color='service_rate',
    color_continuous_scale='Reds',
    scope='usa',
    hover_name='county',
    hover_data={'service_rate': ':.2f', 'avg_crime_rate': ':.2f', 'facility_density': ':.4f', 'fips': False},
    title='California County Mental Health and Crime Analysis (2015-2024)'
)

fig.add_trace(
    px.choropleth(
        county_summary,
        geojson=geojson_url,
        locations='fips',
        color='avg_crime_rate',
        color_continuous_scale='Blues',
        scope='usa',
        hover_name='county',
        hover_data={'avg_crime_rate': ':.2f', 'fips': False},
    ).data[0]
)

fig.add_trace(
    px.choropleth(
        county_summary[~county_summary['county'].isin(['City and County of San Francisco'])],
        geojson=geojson_url,
        locations='fips',
        color='facility_density',
        color_continuous_scale='Greens',
        scope='usa',
        hover_name='county',
        hover_data={'facility_density': ':.4f', 'fips': False},
    ).data[0]
)

fig.update_layout(
    updatemenus=[{
        'buttons': [
            {'label': 'Services per 100k', 'method': 'update',
             'args': [{'visible': [True, False, False]},
                      {'title': 'Mental Health Services per 100k by County (2015-2024)',
                       'coloraxis.colorbar.title.text': 'service_rate'}]},
            {'label': 'Avg Violent Crime per 100k', 'method': 'update',
             'args': [{'visible': [False, True, False]},
                      {'title': 'Average Violent Crime Rate per 100k by County (2015-2024)',
                       'coloraxis.colorbar.title.text': 'avg_crime_rate'}]},
            {'label': 'Facility Density per sq mile', 'method': 'update',
             'args': [{'visible': [False, False, True]},
                      {'title': 'Mental Health Facility Density per Square Mile by County (2015-2024)',
                       'coloraxis.colorbar.title.text': 'facility_density'}]},
        ],
        'direction': 'down',
        'showactive': True,
    }]
)

fig.update_geos(fitbounds="locations")
fig.write_html("ca_map_new.html")
fig.show()

In [ ]:
crimes_avg[crimes_avg['County'] == 'Madera'][['County', 'population', 'avg_violent_sum', 'avg_crime_rate']]

,County,population,avg_violent_sum,avg_crime_rate
16,Madera,160940.0,170.36,105.853113
